# 05 — Classificação de imagens médicas e Grad-CAM com CNN pequena

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flavioluizseixas/aprendizado-de-maquina-para-saude/blob/main/notebooks/05_imagens_gradcam.ipynb)

**Duração estimada:** cerca de 90 minutos  
**Pré-requisitos:** Classificação binária, matrizes e noções iniciais de redes convolucionais.

## Objetivos

- usar as divisões oficiais do PneumoniaMNIST
- treinar uma CNN pequena com validação e early stopping
- avaliar erros e métricas binárias
- gerar Grad-CAM e discutir seus limites

## Fonte e licença

[PneumoniaMNIST/MedMNIST+ — descrição e rótulos](https://github.com/MedMNIST/MedMNIST/blob/main/medmnist/info.py), radiografias pediátricas reduzidas para 64 × 64; 0=normal e 1=pneumonia.

MedMNIST é CC BY 4.0; a fonte original do subconjunto é de Kermany et al. O conjunto não se destina a uso clínico.

> **Uso responsável:** Este material tem finalidade exclusivamente educacional. Os resultados não devem ser usados para diagnóstico, prognóstico, tratamento, gestão assistencial ou decisão de saúde pública sem validação adequada, análise de contexto e supervisão de profissionais qualificados.

## Preparação do ambiente

> Há GPU disponível? A execução continua em CPU com um aviso.

In [ ]:
# Preparação reproduzível do ambiente (a instalação ocorre só se faltar pacote).
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

REPO = "flavioluizseixas/aprendizado-de-maquina-para-saude"
REPO_DIR = Path("/content") / REPO.split("/")[-1]
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    command = ["git", "clone", f"https://github.com/{REPO}.git", str(REPO_DIR)]
    if REPO_DIR.exists():
        command = ["git", "-C", str(REPO_DIR), "pull", "--ff-only"]
    subprocess.run(command, check=True)
    os.chdir(REPO_DIR)
else:
    candidates = [Path.cwd(), Path.cwd().parent]
    project = next((p for p in candidates if (p / "src").exists()), Path.cwd())
    os.chdir(project)

packages = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.1,<4', 'matplotlib': 'matplotlib>=3.8,<4', 'seaborn': 'seaborn>=0.13,<1', 'sklearn': 'scikit-learn>=1.4,<2', 'requests': 'requests>=2.31,<3', 'medmnist': 'medmnist>=3,<4', 'tensorflow': 'tensorflow>=2.16,<3'}
missing = [spec for module, spec in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

from src.config import RANDOM_STATE, seed_everything
seed_everything(RANDOM_STATE)
print(f"Ambiente pronto em {Path.cwd()} | Colab={IN_COLAB} | semente={RANDOM_STATE}")

In [ ]:
import matplotlib.pyplot as plt
import medmnist
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from pathlib import Path
from medmnist import PneumoniaMNIST
from sklearn.metrics import ConfusionMatrixDisplay

from src.data_loading import ensure_medmnist_download
from src.evaluation import classification_report_health, report_frame
from src.gradcam import show_gradcam

FAST_MODE = True
seed_everything(RANDOM_STATE)  # TensorFlow já foi importado nesta célula.
print("GPU:", tf.config.list_physical_devices("GPU") or "não encontrada — usando CPU")

## Pergunta orientadora

> Uma CNN pequena aprende sinais úteis neste benchmark reduzido, e onde a rede concentra influência em acertos e erros?

## Obtenção dos dados

> As divisões oficiais foram preservadas?

In [ ]:
medmnist_root = Path("/content/medmnist" if IN_COLAB else "data/cache/medmnist")
ensure_medmnist_download("pneumoniamnist", 64, medmnist_root)
train_data = PneumoniaMNIST(split="train", download=False, size=64, root=str(medmnist_root))
val_data = PneumoniaMNIST(split="val", download=False, size=64, root=str(medmnist_root))
test_data = PneumoniaMNIST(split="test", download=False, size=64, root=str(medmnist_root))

def arrays(dataset):
    images = np.asarray(dataset.imgs)
    if images.ndim == 3:
        images = images[..., None]
    return images.astype("float32") / 255.0, np.asarray(dataset.labels).ravel().astype(int)

X_train, y_train = arrays(train_data)
X_val, y_val = arrays(val_data)
X_test, y_test = arrays(test_data)

## Inspeção

> Qual é o formato e o equilíbrio das classes?

In [ ]:
for split, X_part, y_part in [("treino", X_train, y_train), ("validação", X_val, y_val), ("teste", X_test, y_test)]:
    counts = pd.Series(y_part).value_counts().sort_index()
    print(split, X_part.shape, counts.to_dict())
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for axis, index in zip(axes.ravel(), np.linspace(0, len(X_train) - 1, 10, dtype=int)):
    axis.imshow(X_train[index].squeeze(), cmap="gray")
    axis.set_title(f"classe {y_train[index]}")
    axis.axis("off")
plt.suptitle("Exemplos do treino — imagens reduzidas e pré-processadas"); plt.tight_layout(); plt.show()

## Preparação e modelo

> Uma arquitetura pequena é suficiente para o objetivo didático?

In [ ]:
if FAST_MODE:
    rng = np.random.default_rng(RANDOM_STATE)
    keep = rng.choice(len(X_train), min(3_000, len(X_train)), replace=False)
    X_fit, y_fit = X_train[keep], y_train[keep]
else:
    X_fit, y_fit = X_train, y_train

model = tf.keras.Sequential([
    tf.keras.layers.Input(X_train.shape[1:]),
    tf.keras.layers.Conv2D(16, 3, activation="relu", padding="same"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(32, 3, activation="relu", padding="same", name="last_conv"),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])

## Experimento

> O desempenho de validação deixa de melhorar antes do limite de épocas?

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_auc", mode="max", patience=2, restore_best_weights=True
)
history = model.fit(
    X_fit, y_fit, validation_data=(X_val, y_val),
    epochs=5 if FAST_MODE else 10, batch_size=64,
    callbacks=[early_stop], verbose=2,
)

In [ ]:
curves = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
curves[["loss", "val_loss"]].plot(ax=axes[0], title="Loss por época")
curves[["auc", "val_auc"]].plot(ax=axes[1], title="AUC por época")
for axis in axes: axis.set_xlabel("Época")
plt.tight_layout(); plt.show()

## Avaliação

> Que erros aparecem no teste oficial?

In [ ]:
probabilities = model.predict(X_test, batch_size=128, verbose=0).ravel()
predictions = (probabilities >= 0.5).astype(int)
report = classification_report_health(y_test, predictions, probabilities)
display(report_frame(report).round(3))
ConfusionMatrixDisplay.from_predictions(y_test, predictions, cmap="Blues", display_labels=["normal", "pneumonia"])
plt.title("Matriz de confusão — teste oficial"); plt.show()

## Grad-CAM

> Onde houve influência para a saída, em acertos e erros?

In [ ]:
categories = {
    "falso positivo": np.where((y_test == 0) & (predictions == 1))[0],
    "falso negativo": np.where((y_test == 1) & (predictions == 0))[0],
    "verdadeiro positivo": np.where((y_test == 1) & (predictions == 1))[0],
    "verdadeiro negativo": np.where((y_test == 0) & (predictions == 0))[0],
}
selected = [(name, int(indices[0])) for name, indices in categories.items() if len(indices)]
if not any(name.startswith("falso") for name, _ in selected):
    print("O modelo não errou neste teste; exibiremos casos limítrofes sem inventar um erro.")
used = {index for _, index in selected}
for index in np.argsort(np.abs(probabilities - 0.5)):
    if int(index) not in used and len(selected) < 4:
        selected.append(("caso limítrofe", int(index)))
selected = selected[:4]
print(selected)

In [ ]:
for name, index in selected:
    probability = probabilities[index]
    print(f"{name}: real={y_test[index]}, probabilidade={probability:.3f}")
    show_gradcam(model, X_test[index], "last_conv", class_index=int(predictions[index]))
    plt.show()

### Como interpretar

Grad-CAM aponta regiões que influenciaram a saída da rede, não uma lesão confirmada nem uma justificativa clínica. Mapas difusos, bordas e artefatos podem revelar atalhos. Acertos não validam o mapa; erros são especialmente informativos.

## Limitações e responsabilidade

- A população é pediátrica e a resolução 64 × 64 remove detalhes da imagem original.
- Equipamento, pré-processamento e atalhos visuais podem mudar o desempenho fora do benchmark.
- Grad-CAM não localiza doença com garantia e não constitui validação clínica.

## Atividade

Escolha dois exemplos com probabilidades próximas de 0,5. Compare os mapas e escreva por que incerteza do modelo e mapa visual respondem a perguntas diferentes.

## Três aprendizados principais

1. Divisões oficiais evitam contaminação entre treino, validação e teste.
2. Matriz de confusão torna falsos positivos e negativos visíveis.
3. Grad-CAM investiga a rede, mas não explica a biologia nem valida uso clínico.

## Referências

- [MedMNIST v2, Scientific Data (2023)](https://www.nature.com/articles/s41597-022-01721-8)
- [MedMNIST no GitHub](https://github.com/MedMNIST/MedMNIST)
- Kermany et al. Identifying Medical Diagnoses and Treatable Diseases by Image-Based Deep Learning. Cell, 2018.
- Selvaraju et al. Grad-CAM. ICCV, 2017.

## Versões das bibliotecas

Registre o ambiente junto ao resultado.

In [ ]:
from src.config import library_versions
library_versions(('numpy', 'pandas', 'matplotlib', 'scikit-learn', 'tensorflow', 'medmnist'))